In [1]:
import sys
sys.path.append("../")
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
import numpy as np
from tqdm import tqdm
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict
from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.config.config import data_settings
import os
import h5py
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import multiprocessing
import time
import csv

In [2]:
data_settings.ChunkSize = 64
data_settings.BatchSize = 5096

def delete_all_files(directory):
    # Check if the directory exists
    if not os.path.exists(directory):
        print(f"The directory {directory} does not exist.")
        return
    
    # Iterate over all files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        try:
            # Check if it's a file (not a subdirectory)
            if os.path.isfile(file_path):
                os.remove(file_path)  # Delete the file
                print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")


In [4]:
def db_to_hdf5_files_single_file(batch_retrieval_size: int = data_settings.BatchSize,
                                chunk_size: int = data_settings.ChunkSize):
    """
    This version creates ONE h5 file for training, ONE for testing, ONE for validation,
    each containing chunked, resizable datasets ('features' and 'labels'),
    with a tqdm progress bar for each data split.
    """

    num_bitboards = len(sample_bitboard_dict.keys())

    sets = {
        data_settings.TrainingDirectory: GamePositionRollup.is_training_data.is_(True),
        data_settings.TestingDirectory: GamePositionRollup.is_testing_data.is_(True),
        data_settings.ValidationDirectory: GamePositionRollup.is_validation_data.is_(True),
    }

    for h5_dir, filter_conditions in sets.items():

        delete_all_files(h5_dir)
        single_file_path = os.path.join(h5_dir, "data_all.h5")

        # Count total records in this split
        with next(get_db()) as session:
            total_records = session.query(GamePositionRollup)\
                                   .filter(filter_conditions)\
                                   .count()
            print(f"[{h5_dir}] Total records: {total_records}")

        if total_records == 0:
            print(f"No records found for {h5_dir}, skipping.")
            continue

        with h5py.File(single_file_path, 'w') as h5f:
            # Create resizable, chunked datasets
            features_dset = h5f.create_dataset(
                "features",
                shape=(0, num_bitboards, 8, 8),
                maxshape=(None, num_bitboards, 8, 8),
                dtype="uint64",
                chunks=(chunk_size, num_bitboards, 8, 8),
                compression="gzip"
            )
            labels_dset = h5f.create_dataset(
                "labels",
                shape=(0, 3),
                maxshape=(None, 3),
                dtype="uint64",
                chunks=(chunk_size, 3),
                compression="gzip"
            )

            current_size = 0

            with next(get_db()) as session:
                # Wrap the main loop with tqdm
                with tqdm(
                    total=total_records, 
                    desc=f"[{h5_dir}] Writing Records", 
                    unit=" records"
                ) as pbar:
                    for batch_start in range(0, total_records, batch_retrieval_size):

                        records = (session.query(GamePositionRollup)
                                  .filter(filter_conditions)
                                  .offset(batch_start)
                                  .limit(batch_retrieval_size)
                                  .all())

                        if not records:
                            break

                        # Collect batch data
                        features_list = []
                        labels_list = []

                        for record in records:
                            # Extract features
                            bitboard_values = [getattr(record, attr) 
                                               for attr in sample_bitboard_dict.keys()]
                            features = bitboards_to_array(bitboard_values)

                            # Extract labels
                            labels = record.win_buckets

                            features_list.append(features)
                            labels_list.append(labels)

                        # Convert to NumPy arrays
                        features_array = np.array(features_list, dtype=np.float32)
                        labels_array   = np.array(labels_list, dtype=np.float32)

                        batch_size = features_array.shape[0]
                        new_size = current_size + batch_size

                        features_dset.resize((new_size, num_bitboards, 8, 8))
                        labels_dset.resize((new_size, 3))

                        features_dset[current_size:new_size, ...] = features_array
                        labels_dset[current_size:new_size, ...]   = labels_array
                        

                        current_size = new_size

                        pbar.update(batch_size)


        print(f"Finished writing dataset to {single_file_path}")

In [5]:
db_to_hdf5_files_single_file()

Deleted: ./src/model/data/training\data_0.h5
[./src/model/data/training] Total records: 2229839


[./src/model/data/training] Writing Records: 100%|█████████████████████████| 2229839/2229839 [05:29<00:00, 6768.51 records/s]


Finished writing dataset to ./src/model/data/training\data_all.h5
Deleted: ./src/model/data/testing\data_0.h5
[./src/model/data/testing] Total records: 46472


[./src/model/data/testing] Writing Records: 100%|██████████████████████████████| 46472/46472 [00:07<00:00, 5857.11 records/s]


Finished writing dataset to ./src/model/data/testing\data_all.h5
Deleted: ./src/model/data/validation\data_0.h5
[./src/model/data/validation] Total records: 46917


[./src/model/data/validation] Writing Records: 100%|███████████████████████████| 46917/46917 [00:08<00:00, 5810.93 records/s]

Finished writing dataset to ./src/model/data/validation\data_all.h5


In [16]:
# Global dictionary {worker_id: h5_file_object}
_worker_h5_handles = {}

def worker_init_fn(worker_id):
    global _worker_h5_handles
    h5_file_path = getattr(torch.utils.data.get_worker_info().dataset, 'h5_path', None)
    print(f"Initializing worker {worker_id} with HDF5 file path: {h5_file_path}")
    if h5_file_path is not None:
        _worker_h5_handles[worker_id] = h5py.File(h5_file_path, 'r', libver='latest', swmr=True)
        print(f"Worker {worker_id} initialized successfully.")



In [17]:
class HDF5SingleFileDataset(Dataset):
    """
    A Dataset that reads from one chunked HDF5 file with datasets:
      - "features" of shape (N, num_bitboards, 8, 8)
      - "labels" of shape (N, 3)
    """
    def __init__(self, h5_path, transform=None):
        """
        Args:
            h5_file_path (str): Path to the .h5 file ('data_all.h5').
            transform (callable, optional): A transform to apply to the features.
        """
        super().__init__()
        self.h5_file_path = f"{h5_path}/data_all.h5"
        self.transform = transform

        # Open once to get length (and optionally shape info)
        with h5py.File(self.h5_file_path, 'r', libver='latest', swmr=True) as h5f:
            self.length = h5f['features'].shape[0]  # number of samples

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Retrieve the worker ID
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            # Single-process data loading (no workers)
            with h5py.File(self.h5_file_path, 'r') as hf:
                features = hf["features"][idx]
                labels   = hf["labels"][idx]
        else:
            # Use the open file handle stored for this worker
            worker_id = worker_info.id
            hf = _worker_h5_handles[worker_id]
            features = hf["features"][idx]
            labels   = hf["labels"][idx]

        # Apply any transform you want to the features
        if self.transform:
            features = self.transform(features)  # for example, normalization, etc.

        # Convert to torch tensors
        features_tensor = torch.from_numpy(features)   # shape: (num_bitboards, 8, 8)
        labels_tensor   = torch.from_numpy(labels)     # shape: (3,)

        return features_tensor, labels_tensor

In [18]:
def get_dataloader(h5_path, batch_size=32,shuffle=True, num_workers=4):
    dataset = HDF5SingleFileDataset(h5_path)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        worker_init_fn=worker_init_fn,
        shuffle=shuffle
    )
    return loader

In [19]:
def get_dataloader_full_retrieval_time():
    num_epochs = 1
    train_loader = get_dataloader(data_settings.TrainingDirectory, batch_size=64, shuffle=True, num_workers=2)
    start = time.time()
    i = 0
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            i = i + features.shape[0]
            # print(f"feature shape: {features.shape}, labels shape: {labels.shape}")
            pass
            # features => shape (64, 12, 8, 8)
            # labels   => shape (64, 3)
            # your training logic here...
    end = time.time()
    elapsed_time = end - start
    print(f"total run time: {elapsed_time}, training examples: {i}")
    return elapsed_time

In [20]:
get_dataloader_full_retrieval_time()

RuntimeError: DataLoader worker (pid(s) 15600, 604) exited unexpectedly